In [1]:
from math import gcd
from functools import reduce
from time import perf_counter
import tarfile
import os
import pandas as pd

In [2]:
# Função que calcula o número de rodadas
def duracao_danca_grande(n, receita):
    if len(receita) != n:
        raise ValueError(f"Tamanho da receita ({len(receita)}) não bate com n ({n})")

    def mmc(a, b):
        return a * b // gcd(a, b)

    def mmc_lista(lista):
        return reduce(mmc, lista)

    visitado = [False] * n
    tamanhos_ciclos = []

    for i in range(n):
        if not visitado[i]:
            atual = i
            tamanho = 0
            while not visitado[atual]:
                visitado[atual] = True
                atual = receita[atual]
                tamanho += 1
            tamanhos_ciclos.append(tamanho)

    return mmc_lista(tamanhos_ciclos)

In [3]:
def processar_tgz_com_tempo(tgz_path, extrair_para='casos_extraidos'):
    # Extrai o .tgz
    with tarfile.open(tgz_path, 'r:gz') as tar:
        tar.extractall(extrair_para)
    
    resultados = []
    for root, _, files in os.walk(extrair_para):
        for nome_arquivo in sorted(files):
            caminho = os.path.join(root, nome_arquivo)
            try:
                n, receita = processar_arquivo_receita(caminho)

                # Cronometrar com alta precisão
                inicio = perf_counter()
                rodadas = duracao_danca_grande(n, receita)
                fim = perf_counter()
                duracao = fim - inicio

                resultados.append({
                    "arquivo": nome_arquivo,
                    "n": n,
                    "rodadas": rodadas,
                    "tempo_segundos": duracao
                })
            except Exception as e:
                resultados.append({
                    "arquivo": nome_arquivo,
                    "n": None,
                    "rodadas": None,
                    "tempo_segundos": None,
                    "erro": str(e)
                })
    
    df_resultados = pd.DataFrame(resultados)
    return df_resultados


In [4]:
# Função que processa um único arquivo de receita
def processar_arquivo_receita(filepath):
    with open(filepath, 'r') as f:
        linhas = f.readlines()
        n = int(linhas[0].strip())
        receita = list(map(int, linhas[1].strip().split()))
        return n, receita

tgz_path = "datasets/casosfinal2.tgz"
df_resultados = processar_tgz_com_tempo(tgz_path)
print(df_resultados)

        arquivo    n        rodadas  tempo_segundos
0   caso102.txt  102       22700678        0.000023
1   caso112.txt  112      380570190        0.000026
2   caso122.txt  122      924241890        0.000020
3   caso132.txt  132     1744004262        0.000025
4   caso142.txt  142     3708514810        0.000026
5   caso152.txt  152    13370699342        0.000021
6   caso162.txt  162    17683828162        0.000021
7   caso172.txt  172   283551037770        0.000030
8   caso182.txt  182   432788426070        0.000023
9   caso192.txt  192  1484147626962        0.000028
10   caso72.txt   72        1939938        0.000012
11   caso82.txt   82        3432198        0.000017
12   caso92.txt   92       14872858        0.000015
